In [2]:
from ultralytics import YOLO
import os
from pathlib import Path
import numpy as np
import yaml

CONF_THRES = 0.01
IOU_THRESHES = np.linspace(0.5, 0.95, 10)

def iou_xyxy(b1, b2):
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    iw = max(0.0, x2 - x1); ih = max(0.0, y2 - y1)
    inter = iw * ih
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1]); a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    union = a1 + a2 - inter + 1e-9
    return inter / union

def compute_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i-1] = np.maximum(mpre[i-1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1])

def load_gt_boxes(label_path, class_ids):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5:
                continue
            cls = int(p[0])
            if cls not in class_ids:
                continue
            xc, yc, w, h = map(float, p[1:5])
            x1 = xc - w/2; y1 = yc - h/2
            x2 = xc + w/2; y2 = yc + h/2
            boxes.append((cls, x1, y1, x2, y2))
    return boxes

def compute_map_for_yaml(model_path, yaml_path, device=0):
    # Leer yaml
    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)

    root = cfg.get("path", ".")
    test_rel = cfg.get("test", cfg.get("val", "images"))
    img_dir = os.path.join(root, test_rel)
    label_dir = img_dir.replace("images", "labels")

    nc = cfg["nc"]
    names = cfg["names"]
    if isinstance(names, dict):
        # convertir {0:name0,...} a lista
        names = [names[i] for i in range(nc)]

    class_ids = list(range(nc))

    print("Modelo:", model_path)
    print("YAML:", yaml_path)
    print("nc:", nc, "names:", names)

    model = YOLO(model_path)

    exts = {".jpg",".jpeg",".png"}
    img_paths = [
        os.path.join(img_dir, f)
        for f in sorted(os.listdir(img_dir))
        if Path(f).suffix.lower() in exts
    ]
    print("Imágenes test:", len(img_paths))

    results = model.predict(
        img_paths,
        imgsz=640,
        device=device,
        conf=CONF_THRES,
        verbose=False,
    )

    preds = {cid: [] for cid in class_ids}
    gts   = {cid: {} for cid in class_ids}

    for img_id, (img_path, r) in enumerate(zip(img_paths, results)):
        img_name = os.path.basename(img_path)
        label_path = os.path.join(
            label_dir, os.path.splitext(img_name)[0] + ".txt"
        )

        gt_boxes = load_gt_boxes(label_path, class_ids)
        for (cls, x1, y1, x2, y2) in gt_boxes:
            gts.setdefault(cls, {})
            gts[cls].setdefault(img_id, [])
            gts[cls][img_id].append((x1, y1, x2, y2))

        if r.boxes is not None:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls not in class_ids:
                    continue
                conf = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxyn[0].tolist()
                preds[cls].append((img_id, conf, (x1, y1, x2, y2)))

    ap50 = {}
    ap5095 = {}

    for cid in class_ids:
        cls_preds = sorted(preds[cid], key=lambda x: -x[1])
        npos = sum(len(b) for b in gts[cid].values()) if cid in gts else 0

        if len(cls_preds) == 0 or npos == 0:
            ap50[cid] = 0.0
            ap5095[cid] = 0.0
            continue

        aps = []
        for iou_th in IOU_THRESHES:
            tp = np.zeros(len(cls_preds))
            fp = np.zeros(len(cls_preds))
            matched = {img_id: np.zeros(len(gts[cid].get(img_id, [])))
                       for img_id in gts[cid].keys()}

            for i, (img_id, conf, bbox_pred) in enumerate(cls_preds):
                gt_boxes = gts[cid].get(img_id, [])
                if len(gt_boxes) == 0:
                    fp[i] = 1
                    continue

                ious = np.array([iou_xyxy(bbox_pred, gt) for gt in gt_boxes])
                j = ious.argmax()
                best_iou = ious[j]
                if best_iou >= iou_th and matched[img_id][j] == 0:
                    tp[i] = 1
                    matched[img_id][j] = 1
                else:
                    fp[i] = 1

            fp_cum = np.cumsum(fp)
            tp_cum = np.cumsum(tp)
            rec = tp_cum / npos
            prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
            aps.append(compute_ap(rec, prec))

        ap50[cid] = aps[0]
        ap5095[cid] = float(np.mean(aps))

    print("\n=== AP por clase ===")
    for cid in class_ids:
        print(f"{cid} {names[cid]:10s} -> "
              f"AP50: {ap50[cid]:.4f}  AP50-95: {ap5095[cid]:.4f}")

    mAP50 = float(np.mean(list(ap50.values())))
    mAP5095 = float(np.mean(list(ap5095.values())))
    print("\n=== mAP global ===")
    print("mAP50:", round(mAP50, 4))
    print("mAP50-95:", round(mAP5095, 4))

    return ap50, ap5095, mAP50, mAP5095

# Ejemplo de uso:
# ap50, ap5095, m50, m5095 = compute_map_for_yaml(
#     "/home/fozamorano/WAID/ModelsWAID/Model_11s_4.pt",
#     "/home/fozamorano/WAID/src/waidplus.yaml",
#     device=2,
# )


In [12]:
from ultralytics import YOLO

model = YOLO("/home/fozamorano/WAID/ModelsWAID/WAID_11n_4.pt")   # tu modelo previo (6 clases)

model.train(
    data="/home/fozamorano/WAID/src/waidplus.yaml",   # 10 clases, train+val definidos completos
    epochs=10,              # aquí decides cuántas pasadas completas al train
    imgsz=640,
    batch=16,
    workers=4,
    device=2,               # o "2" según tu setup
    lr0=0.001,              # el que estabas usando
    freeze=10,              # si quieres conservar bastante WAID
    amp=False,
    val=False,
    project="runs",
    name="waidplus_11n_full",
    seed=42,
)


New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.227 🚀 Python-3.12.9 torch-2.9.0+cu128 CUDA:2 (NVIDIA GeForce RTX 3090, 24260MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/fozamorano/WAID/src/waidplus.yaml, degrees=0.0, deterministic=True, device=2, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/fozamorano/WAID/ModelsWAID/WAID_11n_4.pt, momentum=0.937, mosaic=1.0, mul

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f70f00a3ef0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [1]:
import os
from collections import Counter

# RUTA a tu waidplus.yaml
DATA_YAML = "waidplus.yaml"

import yaml

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

root = cfg.get("path", "")
labels_train = os.path.join(root, cfg["train"].replace("images", "labels"))
labels_val   = os.path.join(root, cfg["val"].replace("images", "labels"))

names = cfg["names"]  # diccionario id->nombre

def count_labels(label_dir):
    counts = Counter()
    n_files = 0
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        n_files += 1
        with open(os.path.join(label_dir, fname), "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
                counts[cls_id] += 1
    return counts, n_files

train_counts, n_train_files = count_labels(labels_train)
val_counts,   n_val_files   = count_labels(labels_val)

print("=== TRAIN ===")
print("Num imágenes con anotaciones:", n_train_files)
for cid, n in sorted(train_counts.items()):
    print(f"{cid:2d} {names[cid]:10s}: {n:6d} instancias")

print("\n=== VAL ===")
print("Num imágenes con anotaciones:", n_val_files)
for cid, n in sorted(val_counts.items()):
    print(f"{cid:2d} {names[cid]:10s}: {n:6d} instancias")


=== TRAIN ===
Num imágenes con anotaciones: 1953
 0 sheep     :   6180 instancias
 1 cattle    :   2688 instancias
 2 seal      :   1198 instancias
 3 camelus   :   1756 instancias
 4 kiang     :   1259 instancias
 5 zebra     :   1638 instancias
 6 crocodile :    267 instancias
 7 elephant  :   5697 instancias
 8 deer      :    184 instancias
 9 horse     :    234 instancias

=== VAL ===
Num imágenes con anotaciones: 402
 0 sheep     :   1972 instancias
 1 cattle    :    799 instancias
 2 seal      :    425 instancias
 3 camelus   :    470 instancias
 4 kiang     :    369 instancias
 5 zebra     :    404 instancias
 6 crocodile :      8 instancias
 7 elephant  :    293 instancias
 8 deer      :      8 instancias
 9 horse     :     22 instancias
